In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import os
import warnings

# ==========================================
# 1. SETUP
# ==========================================
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

BASE_DIR = "/content/sample_data/"
print(f"🚀 Starting Analysis in: {BASE_DIR}")

# IGNORE COLAB DEFAULT FILES
IGNORE_LIST = ["anscombe.json", "california_housing_test.csv", "california_housing_train.csv", "mnist_test.csv", "mnist_train_small.csv", "README.md"]

def get_clean_file_list(pattern):
    raw_files = glob.glob(os.path.join(BASE_DIR, pattern))
    return [f for f in raw_files if os.path.basename(f) not in IGNORE_LIST]

# ==========================================
# 2. DATA LOADER
# ==========================================
def load_satp_metric(file_patterns, col_keywords):
    yearly_totals = {}
    target_files = []
    for pat in file_patterns: target_files.extend(get_clean_file_list(pat))
    target_files = list(set(target_files))

    for f in target_files:
        try:
            dfs = pd.read_excel(f, sheet_name=None) if f.endswith('.xlsx') else {'base': pd.read_csv(f)}
            for _, df in dfs.items():
                df.columns = [str(c).strip().lower() for c in df.columns]

                # Find Year
                year_col = next((c for c in df.columns if 'year' in c), None)
                if not year_col: continue
                df[year_col] = pd.to_numeric(df[year_col], errors='coerce')
                df = df.dropna(subset=[year_col])

                # Filter 'Total' rows
                for col in df.select_dtypes(include=['object']):
                    if 'district' in col or 'state' in col:
                        df = df[~df[col].astype(str).str.contains('total', case=False, na=False)]

                # Find Columns
                target_cols = []
                for col in df.columns:
                    for kw in col_keywords:
                        if kw.lower() in col:
                            target_cols.append(col)
                            break

                if target_cols:
                    for tc in target_cols: df[tc] = pd.to_numeric(df[tc], errors='coerce').fillna(0)
                    sheet_sum = df.groupby(year_col)[target_cols].sum().sum(axis=1)
                    for y, val in sheet_sum.items():
                        yearly_totals[int(y)] = yearly_totals.get(int(y), 0) + val
        except: pass
    return pd.Series(yearly_totals).sort_index()

# ==========================================
# 3. LOAD DATA
# ==========================================

print("\n📊 Loading Data...")
# ACLED
acled_files = get_clean_file_list("ACLED Data*.csv")
if acled_files:
    acled = pd.read_csv(acled_files[0])
    acled['event_date'] = pd.to_datetime(acled['event_date'])
    acled['year'] = acled['event_date'].dt.year
    acled_yearly = acled.groupby('year').agg({'event_id_cnty': 'count', 'fatalities': 'sum'})
else:
    acled = pd.DataFrame()
    acled_yearly = pd.DataFrame()

# SATP
satp_incidents = load_satp_metric(["TERRORISM.xlsx", "MAOIST INCIDENTS.xlsx", "Panel_Data_*.xlsx"], ['number of terrorism', 'total incidents', 'major incidents'])
satp_fatalities = load_satp_metric(["FATALITIES.xlsx", "MAOIST INCIDENTS.xlsx", "Panel_Data_*.xlsx"], ['civilians', 'security forces', 'terrorists', 'total killed'])
satp_arrests = load_satp_metric(["ARRESTS.xlsx", "Panel_Data_*.xlsx"], ['arrest'])
satp_surrenders = load_satp_metric(["*Surrender*.csv", "Panel_Data_*.xlsx"], ['surrender'])
satp_explosions = load_satp_metric(["EXPLOSIONS.xlsx", "Panel_Data_*.xlsx"], ['explosion', 'total_incidents'])
satp_major = load_satp_metric(["MAJOR INCIDENTS.xlsx"], ['total_incidents'])

satp_df = pd.DataFrame({
    'Incidents': satp_incidents, 'Fatalities': satp_fatalities, 'Arrests': satp_arrests,
    'Surrenders': satp_surrenders, 'Explosions': satp_explosions, 'Major_Incidents': satp_major
})
# Safe Reindex to ensure full 2000-2024 range
satp_df = satp_df.reindex(range(2000, 2025), fill_value=0)

# ==========================================
# 4. GENERATE PLOTS
# ==========================================

print("\n🎨 Generating Plots...")

# --- FIG 1: VISUAL COMPARISON (2016-2024 ONLY) ---
# Strict overlap period
common_years = range(2016, 2025)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot Incidents
if not acled_yearly.empty:
    ax1.plot(common_years, acled_yearly.reindex(common_years)['event_id_cnty'], color='#d62728', marker='o', label='ACLED')
ax1.plot(common_years, satp_df.reindex(common_years)['Incidents'], color='#1f77b4', marker='s', label='SATP')
ax1.set_title("Incident Counts (2016-2024)", fontweight='bold')
ax1.set_ylabel("Count")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot Fatalities
if not acled_yearly.empty:
    ax2.plot(common_years, acled_yearly.reindex(common_years)['fatalities'], color='#d62728', marker='o', linestyle='--', label='ACLED')
ax2.plot(common_years, satp_df.reindex(common_years)['Fatalities'], color='#1f77b4', marker='s', linestyle='--', label='SATP')
ax2.set_title("Fatalities (2016-2024)", fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("Figure 1: Visual Comparison of Overlap Period", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('Fig1_Comparison_Overlap.png')


# --- FIG 2: HISTORICAL CONTEXT (2000-2024) ---
# Answering "Trends over time" - The Long View
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(satp_df.index, satp_df['Incidents'], color='navy', linewidth=2, label='SATP Incidents')
ax.fill_between(satp_df.index, satp_df['Incidents'], color='navy', alpha=0.1)

# Highlight the ACLED era
ax.axvspan(2016, 2024, color='red', alpha=0.05, label='ACLED Era (High Volume Disorder)')

ax.set_title("Figure 2: Historical Context - The Decline of Major Conflict (SATP 2000-2024)", fontweight='bold')
ax.set_ylabel("Incidents")
ax.legend()
plt.tight_layout()
plt.savefig('Fig2_Historical_Trends.png')


# --- FIG 3: SATP DEEP DIVE (COMPOSITION) ---
fig, ax = plt.subplots(figsize=(14, 8))
ax.plot(satp_df.index, satp_df['Incidents'], label='Total Incidents', color='navy', linewidth=3)
ax.plot(satp_df.index, satp_df['Arrests'], label='Arrests', color='green', marker='^', linestyle='--')
ax.plot(satp_df.index, satp_df['Surrenders'], label='Surrenders', color='purple', marker='v', linestyle='--')
ax.plot(satp_df.index, satp_df['Explosions'], label='Explosions', color='orange', marker='*')
ax.plot(satp_df.index, satp_df['Major_Incidents'], label='Major Incidents', color='black', marker='x')

ax.set_title("Figure 3: SATP Conflict Composition & State Action", fontweight='bold')
ax.legend(ncol=3)
plt.tight_layout()
plt.savefig('Fig3_SATP_Composition.png')


# --- FIG 4: ACLED COMPOSITION ---
if not acled.empty:
    plt.figure(figsize=(14, 6))
    acled_filtered = acled[acled['year'].isin(common_years)]
    acled_comp = acled_filtered.groupby(['year', 'event_type']).size().unstack(fill_value=0)
    acled_comp.plot(kind='area', stacked=True, cmap='viridis', alpha=0.9, figsize=(14, 6))
    plt.title("Figure 4: ACLED Composition (Protests vs Violence)", fontweight='bold')
    plt.ylabel("Events")
    plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('Fig4_ACLED_Composition.png')


# --- FIG 5: DISTRICT COMPARISON ---
comp_files = get_clean_file_list("SATP_vs_ACLED_District_Comparison(in).csv")
if comp_files:
    comp_df = pd.read_csv(comp_files[0])
    dist_agg = comp_df[comp_df['Year'].isin(common_years)].groupby('District')[['SATP_Incident_Count', 'ACLED_Incident_Count']].sum()

    plt.figure(figsize=(8, 8))
    sns.scatterplot(data=dist_agg, x='SATP_Incident_Count', y='ACLED_Incident_Count', s=60, alpha=0.6)
    mx = max(dist_agg.max())
    plt.plot([0.1, mx], [0.1, mx], 'r--', label="1:1 Agreement")
    plt.xscale('log'); plt.yscale('log')
    plt.title("Figure 5: District Divergence (2016-2024)", fontweight='bold')
    plt.legend()
    plt.tight_layout()
    plt.savefig('Fig5_District_Comparison.png')

print("✅ All visuals generated!")

In [ ]:
import os
from google.colab import files

print("📦 Packaging charts for download...")

# 1. Create a zip file containing all the PNGs
!zip -r analysis_charts.zip *.png

# 2. Trigger the browser download
print("⬇️ Downloading 'analysis_charts.zip'...")
files.download('analysis_charts.zip')